# Step 10.4 — Score → Softmax → Weighted Sum：第一次把 Attention 真正算出来

Step 10.3 已经有三个对象：

```text
e_(u,j)     = decoder step u 与 source position j 的 raw alignment score
α_(u,j)     = normalized attention weight
c_u         = decoder step u 专属的 attention context vector
```

这一节第一次把它们连成精确计算：

```text
raw scores
   ↓ softmax
attention weights
   ↓ weighted sum of encoder states
context vector
```

本节仍然停留在 recurrent encoder–decoder Attention；不使用 Q/K/V 术语，也不进入 Step 10.5。


## 1. Raw Score 为什么不能直接当 Weight？先引入 Softmax

对固定 decoder step `u`，alignment function 会给每个 source position 一个 raw score：

$$
e_u
=
\begin{bmatrix}
e_{u,0} & e_{u,1} & \cdots & e_{u,T_{src}-1}
\end{bmatrix}.
$$

raw score 可以是任意实数，例如：

```text
[2.0, 1.0, 0.0]
```

但 Step 10.3 的 attention weights 需要满足：

$$
\alpha_{u,j}\ge0
$$

以及：

$$
\sum_j\alpha_{u,j}=1.
$$

因此需要把任意 raw scores 转成一组 normalized positive coefficients。

### Exponential `exp(x)`

`exp(x)=e^x` 是 exponential function（指数函数）。这里需要它的两个最基本性质：

- `exp(x)` 永远大于 0；
- x 越大，`exp(x)` 也越大，因此保留 score 的大小顺序。

例如：

```text
exp(2) ≈ 7.389
exp(1) ≈ 2.718
exp(0) = 1
```

### Softmax 定义

对 source position `j`：

$$
\boxed{
\alpha_{u,j}
=
\frac{\exp(e_{u,j})}
{\sum_{k=0}^{T_{src}-1}\exp(e_{u,k})}
}
$$

分子让每个值变成正数；分母把同一个 decoder step 的所有 source positions 一起归一化，因此最终 weights 和为 1。

Softmax 的输入和输出 shape 相同：

```text
scores  : (B,T_src)
weights : (B,T_src)
```

这里只看一个 decoder step。若把多个 decoder steps 堆起来，则可以是 `(B,T_tgt,T_src)`。


## 2. 三个 Source Positions：手算一次 Softmax

固定一个 decoder step，取：

$$
e_u=[2,1,0].
$$

第一步，指数化：

$$
\exp(e_u)
\approx
[7.389,2.718,1.000].
$$

第二步，求和：

$$
Z
=
7.389+2.718+1
\approx
11.107.
$$

这里把分母暂时记作 `Z`，只是为了少重复写同一个和。

第三步，分别除以 Z：

$$
\alpha_u
\approx
[0.665,0.245,0.090].
$$

检查：

$$
0.665+0.245+0.090
\approx
1.
$$

因此最大的 raw score `2` 得到最大的 attention weight，但另外两个位置并没有被强制变成 0。

Softmax 不是“挑最大值”的 hard selection，而是把所有 positions 变成一组相互竞争的 soft weights。因为分母包含所有 positions，一个 score 改变时，其他位置的 normalized weights 也会受到影响。


In [ ]:
import mlx.core as mx

scores = mx.array([2.0, 1.0, 0.0])
weights = mx.softmax(scores, axis=-1)

print('scores:', scores)
print('weights:', weights)
print('weights shape:', weights.shape)
print('sum of weights:', mx.sum(weights))


## 3. Attention Weight 怎样变成 Context？Weighted Sum

现在假设三个 source positions 的 encoder states 都是二维向量：

$$
h_0^{enc}=[1,0],
\qquad
h_1^{enc}=[0,2],
\qquad
h_2^{enc}=[3,1].
$$

刚才得到：

$$
\alpha_u
\approx
[0.665,0.245,0.090].
$$

Attention context vector 定义为 encoder states 的 **weighted sum（加权和）**：

$$
\boxed{
c_u
=
\sum_{j=0}^{T_{src}-1}
\alpha_{u,j}h_j^{enc}
}
$$

逐项计算：

$$
0.665[1,0]
=
[0.665,0]
$$

$$
0.245[0,2]
=
[0,0.490]
$$

$$
0.090[3,1]
=
[0.270,0.090].
$$

相加：

$$
c_u
\approx
[0.935,0.580].
$$

所以 context 不是某一个 encoder state 的简单复制，而是一组 encoder states 按当前 attention distribution 混合得到的向量。

因为 weights 非负且和为 1，这也可以理解成 weighted average（加权平均）。


In [ ]:
encoder_states = mx.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [3.0, 1.0],
])  # (T_src,H_enc)

weights = mx.softmax(scores, axis=-1)  # (T_src,)

# Insert a size-1 axis: (T_src,) -> (T_src,1)
# Broadcasting then scales each encoder-state row by its scalar weight.
weighted_states = weights[:, None] * encoder_states  # (T_src,H_enc)
context = mx.sum(weighted_states, axis=0)              # (H_enc,)

print('encoder_states shape:', encoder_states.shape)
print('weights[:, None] shape:', weights[:, None].shape)
print('weighted_states:', weighted_states)
print('context:', context)
print('context shape:', context.shape)


## 4. Batch Shape：Softmax 沿哪个 Axis？Weighted Sum 又 Reduce 哪个 Axis？

真实 batch 中，对某一个 decoder step：

```text
scores         : (B,T_src)
encoder_states : (B,T_src,H_enc)
```

### `mx.softmax(scores, axis=-1)`

MLX 0.32.2 的 `mx.softmax` 接收 array 和 `axis`。

`axis=-1` 表示最后一个 axis。对 `(B,T_src)` 来说，最后一个 axis 就是 source-position axis。

因此：

```text
(B,T_src)
   ↓ softmax over T_src
(B,T_src)
```

每个 batch row 独立归一化，不会把不同 samples 混在一起。

### 为什么需要 `weights[..., None]`？

weights：

$$
(B,T_{src})
$$

encoder states：

$$
(B,T_{src},H_{enc}).
$$

加入一个 size-1 axis：

$$
(B,T_{src})
\rightarrow
(B,T_{src},1).
$$

然后 broadcasting：

$$
(B,T_{src},1)
\odot
(B,T_{src},H_{enc})
\rightarrow
(B,T_{src},H_{enc}).
$$

最后对 source-position axis 求和：

$$
\operatorname{sum}_{axis=1}
(B,T_{src},H_{enc})
\rightarrow
(B,H_{enc}).
$$

这就是一个 decoder step 的 batch context shape。


In [ ]:
B, T_src, H_enc = 2, 3, 2

batch_scores = mx.array([
    [2.0, 1.0, 0.0],
    [0.0, 1.0, 2.0],
])  # (B,T_src)

batch_encoder_states = mx.array([
    [[1.0, 0.0], [0.0, 2.0], [3.0, 1.0]],
    [[2.0, 1.0], [1.0, 3.0], [0.0, 2.0]],
])  # (B,T_src,H_enc)

batch_weights = mx.softmax(batch_scores, axis=-1)  # (B,T_src)
batch_context = mx.sum(
    batch_weights[..., None] * batch_encoder_states,
    axis=1,
)  # (B,H_enc)

print('batch_scores:', batch_scores.shape)
print('batch_weights:', batch_weights.shape)
print('batch_encoder_states:', batch_encoder_states.shape)
print('batch_context:', batch_context.shape)
print('row sums:', mx.sum(batch_weights, axis=-1))


## 5. 把 Step 10.3 的 Intuition 变成完整 Attention Computation

现在固定 decoder step `u`，整个流程可以精确写成：

### 1. Alignment score

$$
e_{u,j}
=
a(h_{u-1}^{dec},h_j^{enc}).
$$

### 2. Softmax over source positions

$$
\alpha_{u,j}
=
\frac{\exp(e_{u,j})}
{\sum_k\exp(e_{u,k})}.
$$

### 3. Weighted sum

$$
\boxed{
c_u
=
\sum_j\alpha_{u,j}h_j^{enc}
}
$$

因此 shape flow 是：

```text
raw scores
(B,T_src)
   ↓ softmax over source axis
attention weights
(B,T_src)
   ↓ weights[...,None] × encoder_states
(B,T_src,H_enc)
   ↓ sum over source axis
context c_u
(B,H_enc)
```

这就是 Attention 最基本的精确计算。

如果不同 decoder steps 有不同 scores，就会得到不同 `α_u`，进一步得到不同 `c_u`；这正是 Step 10.3 的 dynamic context 现在被真正数值化的方式。

### 两个边界

第一，本节假定 alignment function 已经产生 raw scores，没有继续展开 Bahdanau additive scoring network 的内部参数形式。

第二，实际 softmax 实现会考虑 numerical stability（数值稳定性）；本课程后面会单独讨论稳定 softmax / attention computation。本节只掌握数学定义和 MLX API。

### Step 10.4 材料边界

- raw alignment score `e_(u,j)` 可以是任意实数；softmax 把它变成非负且和为 1 的 attention distribution。
- softmax 在 source-position axis 上做 normalization；输入输出 shape 不变。
- context `c_u` 是 encoder states 的 attention-weighted sum，shape 为 `(B,H_enc)`。
- `weights[...,None]` 增加 size-1 axis，使 `(B,T_src)` 能广播到 `(B,T_src,H_enc)`。
- `mx.sum(..., axis=1)` 把 source-position axis reduce 掉。
- 本节完成第一次精确 score→softmax→weighted-sum Attention computation，但不使用 Q/K/V 术语。
- 本轮只准备 Step 10.4；不进入 Step 10.5。
